## 文本分类：不使用Trainer类训练

上文中我们了解了使用Trainer类微调模型。现在我们尝试不使用这个类

In [70]:
%matplotlib inline

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from datasets import load_from_disk
torch.set_printoptions(edgeitems=2)
torch.manual_seed(123456)

print("")

### 1. 加载数据

In [2]:
ds_file_path = "../../../data/datasets/emotion/embedding_ds"

# 从磁盘中加载数据集
ds = load_from_disk(ds_file_path)

In [3]:
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask', 'hidden_state'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask', 'hidden_state'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask', 'hidden_state'],
        num_rows: 2000
    })
})

**移除不需要的数据集列：**

In [35]:
ds = ds.remove_columns(["text", "hidden_state"])

In [36]:
ds

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
})

### 2. 微调预训练模型准备

#### 2.1 加载预训练模型

In [13]:
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding

In [6]:
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
device

'mps'

In [7]:
# 加载预训练模型
# 我们的数据集标签有6个值
num_labels = 6
checkpoint = "bert-base-uncased"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=num_labels).to(device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [38]:
# 注意我开始移除了["text", "hidden_state"]这2个不需要的列
ds["train"].column_names

['label', 'input_ids', 'token_type_ids', 'attention_mask']

In [39]:
# 我们使用了 DataCollatorWithPadding 作为 data_collator
# 它会自动填充输入序列，使它们具有相同的长度。
# 这样可以确保在训练过程中，每个批次的输入数据具有一致的形状，便于模型处理。
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

#### 2.2 定义数据加载器

In [40]:
train_dataloader = DataLoader(
    # shuffle是打乱，collate_fn 模型会将句子填充到batch中的最大长度
    ds["train"], shuffle=True, batch_size=64, collate_fn=data_collator,
)

valid_dataloader = DataLoader(
    ds["validation"], shuffle=True, batch_size=64, collate_fn=data_collator,
)

> 快速验证一下数据。

In [41]:
batch_count = 0
for batch in valid_dataloader:
    if batch_count % 100 == 0:
        t1 = {k: v.shape for k, v in batch.items()}
        print(t1)
    batch_count += 1
print("valid_bataloader总共有:{}批数据".format(batch_count))

{'input_ids': torch.Size([64, 69]), 'token_type_ids': torch.Size([64, 69]), 'attention_mask': torch.Size([64, 69]), 'labels': torch.Size([64])}
valid_bataloader总共有:32批数据


In [42]:
# 验证数据集总共的行数
ds["validation"].num_rows

2000

In [43]:
# 查看最后一批数据的大小
last_batch_num = batch["labels"].shape[0]
# {k: v.shape for k, v in batch.items()}
last_batch_num

16

In [44]:
# 查看总共数据条数：batch_size * (batch_count - 1) + last_batch_num
64 * (i - 1) + last_batch_num

2000

> 通过上面的执行结果可以看出，Dataloader把验证集总共分成32个批次，最后一个批次大小是16条数据。那么总共的数据刚好是2000条。

把batch传给模型试一下：

#### 2.3 拿一个批次传递给模型试一下

In [47]:
batch = {k: v.to(device) for k, v in batch.items()}

In [50]:
outputs = model(**batch)
type(outputs)

transformers.modeling_outputs.SequenceClassifierOutput

In [51]:
# 查看损失
outputs.loss

tensor(1.7013, device='mps:0', grad_fn=<NllLossBackward0>)

In [52]:
# 查看logits的形状
outputs.logits.shape

torch.Size([16, 6])

我们最后一个批次大小是16，我们的分类有6类。响应的结果维度正确。   
我们计算一下正确率。

In [59]:
true_labels = batch["labels"]
true_labels

tensor([0, 0, 0, 3, 4, 3, 1, 0, 5, 0, 0, 1, 1, 2, 1, 1], device='mps:0')

In [61]:
# 我们模型调用后预测的labels
predicted_labels = torch.argmax(outputs.logits, dim=-1)
predicted_labels

tensor([0, 0, 0, 0, 2, 2, 2, 0, 0, 0, 2, 2, 2, 0, 2, 0], device='mps:0')

我们肉眼可见的这个模型，预测的分类错误率很高。

**现在我们来计算一下正确率：**

In [85]:
batch["labels"]

tensor([1, 1, 1, 1, 5, 1, 2, 0, 0, 1, 4, 5, 1, 4, 0, 0, 1, 0, 1, 3, 1, 0, 1, 1,
        4, 0, 0, 2, 4, 3, 4, 1, 4, 1, 2, 4, 2, 4, 1, 0, 1, 0, 3, 1, 0, 3, 0, 0,
        5, 3, 1, 1, 1, 0, 3, 1, 1, 3, 1, 1, 5, 2, 4, 3], device='mps:0')

In [86]:
outputs.logits.argmax(-1)

tensor([1, 1, 1, 1, 4, 1, 2, 0, 0, 1, 4, 1, 1, 4, 0, 0, 1, 0, 1, 3, 1, 0, 1, 1,
        4, 0, 0, 2, 4, 3, 4, 1, 4, 1, 2, 4, 2, 4, 1, 0, 1, 0, 3, 1, 0, 3, 0, 0,
        5, 3, 1, 1, 1, 0, 3, 1, 1, 3, 1, 1, 5, 1, 4, 0], device='mps:0')

In [87]:
import evaluate

accuracy_metric = evaluate.load("accuracy")

# 计算指标
preds = outputs.logits.argmax(-1)
accuracy_metric.compute(predictions=preds, references=batch["labels"])

{'accuracy': 0.9375}

#### 2.4 优化器

In [64]:
from torch.optim import AdamW

In [65]:
optimizer = AdamW(model.parameters(), lr=5e-5)

In [66]:
from transformers import get_scheduler

In [69]:
num_epochs = 3  # 训练的次数

num_training_steps = num_epochs * len(train_dataloader)   # 训练步数，每一个批次数据就是算一步

# 获取学习率的调度器：学习率调度器在训练过程中动态调整学习率，以便更好地控制模型的收敛行为
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,  # 优化器
    num_warmup_steps=0,
    num_training_steps=num_training_steps,  # 训练的步数
)

# 查看总共要训练的steps数
print("num_training_steps:", num_training_steps)

num_training_steps: 750


我们训练总共的步数是`750`. 这里我们用到了transformers`的`get_scheduler()`获取学习率调度器的函数。

学习率调度器在训练过程中动态调整学习率，以便更好地控制模型的收敛行为。

学习率调度器的作用有：
1. **提高训练侠侣**：通过在训练过程中调整学习率，可以更快地找到模型的最优参数。
2. **避免陷入局部最优解**：动态调整学习率可以帮助模型跳出局部最优解，找到更好的全局最优解。
3. **减少过拟合**：再训练的后期降低学习率，可以使模型的参数更稳定，从而减少过拟合的风险。

In [68]:
optimizer

AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    initial_lr: 5e-05
    lr: 5e-05
    maximize: False
    weight_decay: 0.01
)

### 3. 训练循环

> 我们在加载模型的时候，就判断了是使用`cuda`还是`mps`还是`cpu`。并且使用了`model.to(device)`。

#### 3.1 训练模型

In [77]:
# 实例化进度条
process_bar = tqdm(range(num_training_steps))

# 把模型置于训练模式
model.train()

# 训练次数
i = 0

for epoch in range(num_epochs):
    # 每次都把训练数据的批次一个一个取出来训练
    for batch in train_dataloader:
        i += 1
        
        # 每个批次的数据，加载到device中
        batch = {k: v.to(device) for k, v in batch.items()}
        # 执行模型：前向传播
        outputs = model(**batch)
        
        # 获取到损失
        loss = outputs.loss
        # 执行反向传播: 更新params的梯度
        loss.backward()

        # 执行优化器的step()：这样不需要手动去修改模型的params
        optimizer.step()

        # 更新学习率
        lr_scheduler.step()

        # 批次操作完，对模型参数的梯度置零
        optimizer.zero_grad()

        # 进度条向前迈进一步
        process_bar.update(1)

        # 输出信息
        if (i - 1) % 100 == 0:
            print("第{}次训练：Loss is {:.2f}".format(i, float(loss)))

# 最后也再输出一次信息：
print("第{}次训练：Loss is {:.2f}".format(i, float(loss)))

  0%|          | 0/750 [00:00<?, ?it/s]

第1次训练：Loss is 0.74
第101次训练：Loss is 0.36
第201次训练：Loss is 0.19
第301次训练：Loss is 0.06
第401次训练：Loss is 0.10
第501次训练：Loss is 0.09
第601次训练：Loss is 0.08
第701次训练：Loss is 0.18
第750次训练：Loss is 0.09


#### 3.2 评估模型

> 我们继续使用`Evaluate`库提供的指标。`metric.compute()`可直接计算。
> 但是当我们在一个预测循环中时，我们需要累积每个批次的`batch`的结果，那么我们需要用到`metric.add_batch()`方法。
> 累积了所有`batch`，就再使用`metric.compute()`获得最终的结果。

In [88]:
metric = evaluate.load("accuracy")

# 设置模型为推理模式
model.eval()

for batch in valid_dataloader:
    # 数据加载到device
    batch = {k: v.to(device) for k, v in batch.items()}

    # 在模型的推理阶段，我们只需要前向传播：不计算梯度，减少内存消耗，加快计算速度。
    with torch.no_grad():
        outputs = model(**batch)

    # 推理的结果
    logits = outputs.logits
    # 预测的标签值
    preds = torch.argmax(logits, dim=-1)
    # 使用指标的add_batch方法
    metric.add_batch(predictions=preds, references=batch["labels"])

# 验证数据集执行完毕，计算指标
metric.compute()

{'accuracy': 0.936}

我们发现正确率是`93.6%`，我们只训练了3轮。准确率还不错。

### 4. 保存模型

In [92]:
model_output_dir = "../../../models/train/emotion"
model_save_dir = f"{model_output_dir}/model-classfication02"

# 保存模型
model.save_pretrained(model_save_dir)

# 分词器也需要保存：要不pipeline使用的时候会报分词器错误：Can't load tokenizer
tokenizer.save_pretrained(model_save_dir)

('../../../models/train/emotion/model-classfication02/tokenizer_config.json',
 '../../../models/train/emotion/model-classfication02/special_tokens_map.json',
 '../../../models/train/emotion/model-classfication02/vocab.txt',
 '../../../models/train/emotion/model-classfication02/added_tokens.json',
 '../../../models/train/emotion/model-classfication02/tokenizer.json')

In [93]:
!ls {model_save_dir}

config.json             special_tokens_map.json tokenizer_config.json
model.safetensors       tokenizer.json          vocab.txt
